In [1]:
import pandas as pd
import json
from openai import OpenAI
from typing import Any, List, Tuple
from beartype import beartype
from autoddg.evaluation import BaseEvaluator, PreferenceEvaluator

/home/bia/miniconda3/envs/nlp-final/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_CONFIG = {
    "base_url": "http://localhost:11434/v1",
    "api_key": "ollama",
    # "model_name": "llama3.1:70b-instruct-q3_K_M",
    "model_name": "llama3.1:8b",
}

In [8]:
eval_path = "results-systematic-eval.csv"

In [9]:
df = pd.read_csv(eval_path)

In [10]:
class PreferenceCaller:
    """A wrapper class to set up and manage the PreferenceEvaluator."""
    def __init__(self, model_name: str = MODEL_CONFIG["model_name"]):
        client = OpenAI(
            api_key=MODEL_CONFIG["api_key"], 
            base_url=MODEL_CONFIG["base_url"]
        )
        # Instantiate the specific evaluator for comparison
        self.evaluator = PreferenceEvaluator(client=client, model_name=model_name)
        
# Modified method in PreferenceCaller:
    def run_preference_evaluation(self, description_a: str, description_b: str) -> dict:
        """Calls the LLM and attempts to parse the JSON response, storing raw output."""
        
        print("-> Calling LLM for preference...")
        # Ensure system message is accessible (assuming it's set in __init__)
        self.evaluator._system_message = self.evaluator._system_message 
        raw_response = self.evaluator.evaluate(description_a, description_b)
        
        # --- JSON Cleaning Block ---
        cleaned_response = raw_response.strip()
        cleaned_response = cleaned_response.replace('\u00A0', ' ').strip()
        cleaned_response = cleaned_response.strip('` \n\t') 
        if cleaned_response.startswith('```') and cleaned_response.endswith('```'):
            cleaned_response = cleaned_response.strip('`').strip()
        # --- END JSON Cleaning Block ---
        
        evaluation_result = {
            'Preference': 'Error', 
            'Score_A': None, 
            'Score_B': None, 
            'Rationale': 'JSON Decode Failure or Unhandled Error',
            # ALWAYS store the original, uncleaned output
            'Raw_LLM_Response': raw_response 
        }
        
        try:
            # Attempt to decode the cleaned response
            parsed_data = json.loads(cleaned_response)
            
            # Update the result dictionary with successfully parsed data
            evaluation_result['Preference'] = parsed_data.get('Preference', 'N/A')
            evaluation_result['Score_A'] = parsed_data.get('Score_A', None)
            evaluation_result['Score_B'] = parsed_data.get('Score_B', None)
            evaluation_result['Rationale'] = parsed_data.get('Rationale', 'N/A')
            # If successful, we can clear the generic failure rationale
            evaluation_result['Rationale'] = parsed_data.get('Rationale', 'Successfully parsed.')
            
        except json.JSONDecodeError as e:
            print(f"🚨 ERROR: Failed to decode JSON response: {e}")
            print(f"Raw Response (Cleaned): {cleaned_response[:200]}...")
            # The 'Raw_LLM_Response' is already preserved in evaluation_result
            
        return evaluation_result

In [13]:
import os
import pandas as pd
import json
from datetime import datetime
from typing import Dict, Any

# Define the columns used in the output CSV
OUTPUT_COLUMNS = [
    'Dataset_Name', 'Prompt_Type_B', 'Comparison_A', 'Comparison_B', 
    'Preference', 'Score_A', 'Score_B', 'Rationale', 'Raw_LLM_Response',
    'Evaluation_Timestamp' # Added a timestamp for consistency
]

def log_preference_result(
    dataset_name: str,
    prompt_type_b: str,
    comparison_a: str,
    comparison_b: str,
    evaluation_result: Dict[str, Any], 
    file_path: str
) -> None:
    """Logs the results of a single preference comparison to a CSV file."""
    
    # Safely cast potentially complex LLM outputs to strings before cleaning
    rationale_str = str(evaluation_result.get('Rationale', ''))
    raw_response_str = str(evaluation_result.get('Raw_LLM_Response', ''))

    new_row = {
        'Dataset_Name': dataset_name,
        'Prompt_Type_B': prompt_type_b,
        'Comparison_A': comparison_a,
        'Comparison_B': comparison_b,
        'Preference': evaluation_result.get('Preference'),
        'Score_A': evaluation_result.get('Score_A'),
        'Score_B': evaluation_result.get('Score_B'),
        # Now using the safely cast strings
        'Rationale': rationale_str.replace('\n', ' '), 
        'Raw_LLM_Response': raw_response_str.replace('\n', ' '), 
        'Evaluation_Timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    }

    # 2. Convert to DataFrame and save
    df_new = pd.DataFrame([new_row], columns=OUTPUT_COLUMNS)
    
    # Check if file exists to decide whether to write header
    header_needed = not os.path.exists(file_path)
    
    # Append to CSV
    df_new.to_csv(file_path, mode='a', header=header_needed, index=False)
    print(f"✅ Logged preference: {dataset_name}")

In [14]:
# --- Instantiate the Caller ---
pref_caller = PreferenceCaller()
OUTPUT_FILE = 'results-systematic-preference.csv'

# --- 🎯 Step-by-Step Execution Plan ---

# 1. Isolate the Baseline Description
baseline_df = df[df['Description_Source'] == 'Vanilla_AutoDDG'].copy()
baseline_desc_map = baseline_df.set_index('Dataset_Name')['Description_Text'].to_dict()

# 2. Isolate the Descriptions to be Compared (Augmented)
augmented_df = df[df['Description_Source'] == 'Augmented_AutoDDG'].copy()

# 3. Create a New List to Store Results
comparison_results = []

# 4. Iterate and Compare
# Iterate over the augmented descriptions (our "Method B")
for index, row in augmented_df.iterrows():
    
    dataset_name = row['Dataset_Name']
    prompt_type = row['Prompt_Type'] # This is the Prompt_Type_B
    
    description_a_vanilla = baseline_desc_map.get(dataset_name)
    description_b_augmented = row['Description_Text']
    
    if not description_a_vanilla:
        print(f"Warning: No Vanilla_AutoDDG found for {dataset_name}. Skipping.")
        continue
    
    # 1. Call the Preference Evaluator (the only external call)
    evaluation_result = pref_caller.run_preference_evaluation(
        description_a=description_a_vanilla,
        description_b=description_b_augmented
    )
    
    # 2. Log the result immediately to the CSV file
    log_preference_result(
        dataset_name=dataset_name,
        prompt_type_b=prompt_type,
        comparison_a='Vanilla_AutoDDG',
        comparison_b='Augmented_AutoDDG',
        evaluation_result=evaluation_result,
        file_path=OUTPUT_FILE
    )

print("\n--- Incremental Preference Evaluation Complete ---")
print(f"Results are saved in {OUTPUT_FILE}")

-> Calling LLM for preference...
✅ Logged preference: The FluPRINT database
-> Calling LLM for preference...
✅ Logged preference: The FluPRINT database
-> Calling LLM for preference...
✅ Logged preference: The FluPRINT database
-> Calling LLM for preference...
✅ Logged preference: The FluPRINT database
-> Calling LLM for preference...
✅ Logged preference: The FluPRINT database
-> Calling LLM for preference...
✅ Logged preference: The FluPRINT database
-> Calling LLM for preference...
✅ Logged preference: CODE-15%: a large scale annotated dataset of 12-lead ECGs
-> Calling LLM for preference...
✅ Logged preference: CODE-15%: a large scale annotated dataset of 12-lead ECGs
-> Calling LLM for preference...
✅ Logged preference: CODE-15%: a large scale annotated dataset of 12-lead ECGs
-> Calling LLM for preference...
✅ Logged preference: CODE-15%: a large scale annotated dataset of 12-lead ECGs
-> Calling LLM for preference...
✅ Logged preference: CODE-15%: a large scale annotated dataset 